In [52]:
import os, sys

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.12.101-hotspot"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]
# garante que driver e workers usem o MESMO interpretador (erro clássico no Windows)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("telecom-silver")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.session.timeZone", "America/Sao_Paulo")
    .config("spark.sql.shuffle.partitions", "8")   # default é 200: absurdo p/ 0,5M linhas
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

Spark 4.2.0


In [53]:
from pathlib import Path

BRONZE = Path("dados_telecom").resolve().as_posix()
SILVER = Path("silver").resolve().as_posix()

def ler(nome):
    return (spark.read
            .option("header", True)
            .option("inferSchema", False)   # proposital: ver explicação
            .option("encoding", "UTF-8")
            .csv(f"{BRONZE}/{nome}.csv"))

tabelas = ["dim_regiao", "clientes_operadoras", "tipos_servico", "contratos_sla",
           "tecnicos", "ctos", "alarmes_rede", "medicoes_potencia",
           "assinantes", "ordens_servico"]

bronze = {t: ler(t) for t in tabelas}
for t, df in bronze.items():
    print(f"{t:22} {df.count():>7} linhas")

dim_regiao                   7 linhas
clientes_operadoras          4 linhas
tipos_servico                7 linhas
contratos_sla               30 linhas
tecnicos                    41 linhas
ctos                      1200 linhas
alarmes_rede             31574 linhas
medicoes_potencia       438000 linhas
assinantes               28952 linhas
ordens_servico           50600 linhas


In [54]:
os_raw  = bronze["ordens_servico"]
ass_raw = bronze["assinantes"]

def dedup(df, chave, ordem):
    """Mantém 1 linha por chave, a mais recente por `ordem`."""
    w = Window.partitionBy(chave).orderBy(F.col(ordem).desc_nulls_last())
    return (df.withColumn("_rn", F.row_number().over(w))
              .filter(F.col("_rn") == 1)
              .drop("_rn"))

os_dedup  = dedup(os_raw, "id_os", "dt_atualizacao")
ass_dedup = dedup(ass_raw, "id_assinante", "data_ativacao")

print("OS:         ", os_raw.count(),  "->", os_dedup.count())
print("Assinantes: ", ass_raw.count(), "->", ass_dedup.count())

OS:          50600 -> 50000
Assinantes:  28952 -> 28779


In [55]:
COLS_DATA = ["dt_abertura", "dt_despacho", "dt_chegada",
             "dt_encerramento", "dt_cancelamento", "dt_atualizacao"]

FMT_ISO = "yyyy-MM-dd HH:mm:ss"
FMT_BR  = "dd/MM/yyyy HH:mm"

def ts(c):
    """Tenta ISO; se nao casar, tenta BR. try_* devolve NULL em vez de estourar (ANSI)."""
    return F.coalesce(
        F.try_to_timestamp(F.col(c), F.lit(FMT_ISO)),
        F.try_to_timestamp(F.col(c), F.lit(FMT_BR)),
    )

os_dt = os_dedup
for c in COLS_DATA:
    os_dt = os_dt.withColumn(c, ts(c))

# controle: data veio preenchida mas nenhum dos dois formatos leu? Tem que dar 0 em tudo.
(os_dedup.agg(*[
    F.sum(F.when(ts(c).isNull() & F.col(c).isNotNull(), 1).otherwise(0)).alias(c)
    for c in COLS_DATA
]).show(truncate=False))

# quantas vieram em formato BR (confere com _metadados.json: 750)
(os_dedup.agg(F.sum(F.when(
    F.try_to_timestamp(F.col("dt_abertura"), F.lit(FMT_ISO)).isNull()
    & F.col("dt_abertura").isNotNull(), 1).otherwise(0)).alias("formato_br")).show())

+-----------+-----------+----------+---------------+---------------+--------------+
|dt_abertura|dt_despacho|dt_chegada|dt_encerramento|dt_cancelamento|dt_atualizacao|
+-----------+-----------+----------+---------------+---------------+--------------+
|0          |0          |0         |0              |0              |0             |
+-----------+-----------+----------+---------------+---------------+--------------+

+----------+
|formato_br|
+----------+
|       750|
+----------+



In [56]:
ACENTOS_DE = "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ"
ACENTOS_PARA = "AAAAAEEEEIIIIOOOOOUUUUCN"

def normalizar(c):
    """trim -> maiúsculo -> sem acento -> espaços internos colapsados."""
    x = F.upper(F.trim(F.col(c)))
    x = F.translate(x, ACENTOS_DE, ACENTOS_PARA)
    return F.regexp_replace(x, r"\s+", " ")

os_norm = (os_dt
    # o código do serviço vira a versão canônica: rep_ftth -> REP_FTTH
    .withColumn("codigo_tipo_servico", F.upper(F.trim(F.col("codigo_tipo_servico"))))
    # o bairro original é preservado; a versão normalizada é a que serve para agrupar
    .withColumn("bairro_norm", normalizar("bairro"))
)

# confirmação: agora só devem existir 7 códigos
os_norm.groupBy("codigo_tipo_servico").count().orderBy(F.desc("count")).show()

+-------------------+-----+
|codigo_tipo_servico|count|
+-------------------+-----+
|          INST_FTTH|16779|
|           REP_FTTH|15575|
|           REP_LENT| 6009|
|            MUD_END| 3807|
|            RET_EQP| 3704|
|           MAN_PREV| 2376|
|           REP_ROMP| 1750|
+-------------------+-----+



In [57]:
ctos = (bronze["ctos"]
        .select("id_cto", "id_olt", "id_regiao",
                F.col("bairro").alias("bairro_cto"),
                F.col("cidade").alias("cidade_cto"),
                F.col("uf").alias("uf_cto"),
                F.col("latitude").cast("double"),
                F.col("longitude").cast("double")))

os_loc = (os_norm.join(ctos.select("id_cto", "bairro_cto", "cidade_cto", "uf_cto"),
                       on="id_cto", how="left")
    .withColumn("flag_bairro_divergente",
                F.col("bairro_norm") != normalizar("bairro_cto"))
    # a CTO manda; o bairro da OS só entra se a OS não tiver CTO
    .withColumn("bairro_confiavel", F.coalesce(F.col("bairro_cto"), F.col("bairro")))
)

os_loc.filter("flag_bairro_divergente").count()

0

In [58]:
trocado = F.col("dt_encerramento") < F.col("dt_chegada")

os_fix = (os_loc
    .withColumn("flag_datas_trocadas", F.coalesce(trocado, F.lit(False)))
    .withColumn("_chegada_ok",     F.when(trocado, F.col("dt_encerramento")).otherwise(F.col("dt_chegada")))
    .withColumn("_encerramento_ok", F.when(trocado, F.col("dt_chegada")).otherwise(F.col("dt_encerramento")))
    .drop("dt_chegada", "dt_encerramento")
    .withColumnRenamed("_chegada_ok", "dt_chegada")
    .withColumnRenamed("_encerramento_ok", "dt_encerramento")
)

os_fix.filter("flag_datas_trocadas").count()   # esperado: 133

133

In [59]:
def horas(fim, ini):
    return F.round((F.col(fim).cast("long") - F.col(ini).cast("long")) / 3600.0, 3)

os_tempos = (os_fix
    .withColumn("h_ate_despacho",   horas("dt_despacho", "dt_abertura"))
    .withColumn("h_deslocamento",   horas("dt_chegada", "dt_despacho"))
    .withColumn("h_execucao",       horas("dt_encerramento", "dt_chegada"))   # TMA em campo
    .withColumn("h_atendimento",    horas("dt_encerramento", "dt_abertura"))  # o relógio do SLA
    .withColumn("flag_tempo_negativo",
                (F.col("h_ate_despacho") < 0) | (F.col("h_deslocamento") < 0) |
                (F.col("h_execucao") < 0) | (F.col("h_atendimento") < 0))
)

os_tempos.select("h_ate_despacho","h_deslocamento","h_execucao","h_atendimento").summary().show()

+-------+-----------------+-------------------+------------------+------------------+
|summary|   h_ate_despacho|     h_deslocamento|        h_execucao|     h_atendimento|
+-------+-----------------+-------------------+------------------+------------------+
|  count|            47431|              47431|             47429|             47429|
|   mean|39.87729314161651| 1.3365764162678386|1.6972647747158935|42.912746779396805|
| stddev|42.19376938470733|0.47993128322473033|1.1030906498384312| 42.17859750541743|
|    min|             0.05|              0.091|             0.083|             0.969|
|    25%|           15.838|              0.963|             0.991|            18.659|
|    50%|           23.336|              1.328|             1.499|            26.424|
|    75%|           48.272|              1.691|             2.163|            51.537|
|    max|           805.77|              3.814|            17.022|           810.803|
+-------+-----------------+-------------------+-------

In [60]:
# ============================================================
# PASSO 9 - Cruzar com o SLA vigente
# ============================================================
ct = (bronze["contratos_sla"]
      .withColumn("sla_horas", F.col("sla_horas").cast("int"))
      .withColumn("valor_os", F.col("valor_os").cast("double"))
      .withColumn("multa_por_hora_atraso", F.col("multa_por_hora_atraso").cast("double"))
      .withColumn("teto_multa_pct", F.col("teto_multa_pct").cast("double"))
      .withColumn("vigencia_inicio", F.to_date("vigencia_inicio"))
      .withColumn("vigencia_fim", F.to_date("vigencia_fim")))

o = os_tempos.alias("o")
c = ct.alias("c")

cond = (
    (F.col("o.id_cliente") == F.col("c.id_cliente")) &
    (F.col("o.codigo_tipo_servico") == F.col("c.codigo_tipo_servico")) &
    (F.to_date("o.dt_abertura") >= F.col("c.vigencia_inicio")) &
    (F.col("c.vigencia_fim").isNull() | (F.to_date("o.dt_abertura") <= F.col("c.vigencia_fim")))
)

os_sla = (o.join(c, cond, "left")
    .select("o.*", "c.id_contrato", "c.versao", "c.sla_horas",
            "c.valor_os", "c.multa_por_hora_atraso", "c.teto_multa_pct"))

print("linhas apos join:", os_sla.count(), "| esperado: 50000")
print("sem contrato:", os_sla.filter("sla_horas is null").count(), "| esperado: 0")

linhas apos join: 50000 | esperado: 50000
sem contrato: 0 | esperado: 0


In [61]:
# ============================================================
# PASSO 10 - Afericao de SLA e multa
# ============================================================

encerrada = F.col("dt_encerramento").isNotNull()

# Regra de negocio: so OS CONCLUIDA e encerrada entra no indicador.
# CANCELADA nao foi executada; IMPRODUTIVA e falha do cliente (ausente);
# EM_ABERTO ainda esta rodando. As tres ficam na base, mas fora do denominador.
conta = encerrada & (F.col("status") == "CONCLUIDA")

os_kpi = (os_sla
    .withColumn("conta_para_sla", conta)

    .withColumn("dentro_sla",
                F.when(conta, F.col("h_atendimento") <= F.col("sla_horas")))

    .withColumn("horas_atraso",
                F.when(conta, F.greatest(F.lit(0.0),
                                         F.col("h_atendimento") - F.col("sla_horas"))))

    # ATENCAO: least() IGNORA nulos em vez de propagar. Sem o when() externo,
    # toda OS sem afericao receberia valor_os * teto como multa fantasma.
    .withColumn("multa_estimada",
                F.when(conta,
                       F.round(F.least(F.col("horas_atraso") * F.col("multa_por_hora_atraso"),
                                       F.col("valor_os") * F.col("teto_multa_pct")), 2)))

    # marca quando o teto contratual limitou a multa (metrica util na gold)
    .withColumn("multa_no_teto",
                F.when(F.col("multa_estimada") > 0,
                       F.abs(F.col("multa_estimada")
                             - F.round(F.col("valor_os") * F.col("teto_multa_pct"), 2)) < 0.01))
)

os_kpi.cache()   # varias acoes a seguir; evita recalcular a cadeia toda

# ---------- conferencias ----------
print("=== 1. multa so pode existir onde conta_para_sla = true ===")
(os_kpi.groupBy("status")
   .agg(F.count("*").alias("qtd"),
        F.sum(F.when(F.col("multa_estimada").isNotNull(), 1).otherwise(0)).alias("multa_nao_nula"),
        F.round(F.sum("multa_estimada"), 2).alias("soma"))
   .orderBy("status").show(truncate=False))

print("=== 2. soma da coluna inteira deve bater com a soma filtrada ===")
os_kpi.agg(F.round(F.sum("multa_estimada"), 2).alias("coluna_inteira")).show()

print("=== 3. KPI por cliente ===")
(os_kpi.filter("conta_para_sla")
   .groupBy("id_cliente")
   .agg(F.count("*").alias("os"),
        F.round(F.avg(F.col("dentro_sla").cast("int")) * 100, 1).alias("pct_no_sla"),
        F.round(F.sum("multa_estimada"), 2).alias("multa"))
   .orderBy("id_cliente").show())

print("=== 4. quantas multas bateram no teto contratual ===")
os_kpi.filter("multa_estimada > 0").groupBy("multa_no_teto").count().show()

=== 1. multa so pode existir onde conta_para_sla = true ===
+-----------+-----+--------------+---------+
|status     |qtd  |multa_nao_nula|soma     |
+-----------+-----+--------------+---------+
|CANCELADA  |2343 |0             |NULL     |
|CONCLUIDA  |44374|44374         |377490.81|
|EM_ABERTO  |228  |0             |NULL     |
|IMPRODUTIVA|3055 |0             |NULL     |
+-----------+-----+--------------+---------+

=== 2. soma da coluna inteira deve bater com a soma filtrada ===
+--------------+
|coluna_inteira|
+--------------+
|     377490.81|
+--------------+

=== 3. KPI por cliente ===
+----------+-----+----------+---------+
|id_cliente|   os|pct_no_sla|    multa|
+----------+-----+----------+---------+
|     CLI01|17880|      60.6|224104.83|
|     CLI02|12215|      80.3| 75281.36|
|     CLI03| 8622|      80.4| 52242.33|
|     CLI04| 5657|      87.5| 25862.29|
+----------+-----+----------+---------+

=== 4. quantas multas bateram no teto contratual ===
+-------------+-----+
|mult

In [62]:
# ============================================================
# PASSO 11 - Gravar a silver (via pandas; contorna o winutils)
# ============================================================
import os as _os
import shutil

SILVER = Path("silver").resolve().as_posix()
_os.makedirs(SILVER, exist_ok=True)

os_silver = os_kpi.withColumn("ano_mes", F.date_format("dt_abertura", "yyyy-MM"))

def gravar(df, nome, particoes=None):
    """Traz para o driver e grava com pyarrow. Viavel porque o volume e pequeno."""
    destino = f"{SILVER}/{nome}"
    shutil.rmtree(destino, ignore_errors=True)   # sem isto, to_parquet ACRESCENTA
    pdf = df.toPandas()
    if particoes:
        pdf.to_parquet(destino, partition_cols=particoes)
    else:
        _os.makedirs(destino, exist_ok=True)
        pdf.to_parquet(f"{destino}/part-0.parquet")
    print(f"{nome:22} {len(pdf):>7} linhas")
    return pdf

gravar(os_silver, "ordens_servico", particoes=["ano_mes"])
for nome in ["dim_regiao", "clientes_operadoras", "tipos_servico", "tecnicos"]:
    gravar(bronze[nome], nome)
gravar(ctos, "ctos")

c:\Users\filip\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\filip\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\filip\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


ordens_servico           50000 linhas
dim_regiao                   7 linhas
clientes_operadoras          4 linhas


c:\Users\filip\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\filip\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\filip\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


tipos_servico                7 linhas
tecnicos                    41 linhas
ctos                      1200 linhas


c:\Users\filip\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\filip\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,id_cto,id_olt,id_regiao,bairro_cto,cidade_cto,uf_cto,latitude,longitude
0,CTO-RJ-CAP-N-0001,OLT-RJ-CAP-N-A,RJ-CAP-N,Madureira,Rio de Janeiro,RJ,-22.888131,-43.246788
1,CTO-RJ-CAP-N-0002,OLT-RJ-CAP-N-B,RJ-CAP-N,Penha,Rio de Janeiro,RJ,-22.858848,-43.311375
2,CTO-RJ-CAP-N-0003,OLT-RJ-CAP-N-A,RJ-CAP-N,Madureira,Rio de Janeiro,RJ,-22.837350,-43.294515
3,CTO-RJ-CAP-N-0004,OLT-RJ-CAP-N-B,RJ-CAP-N,Tijuca,Rio de Janeiro,RJ,-22.826908,-43.304488
4,CTO-RJ-CAP-N-0005,OLT-RJ-CAP-N-A,RJ-CAP-N,Méier,Rio de Janeiro,RJ,-22.874086,-43.291561
...,...,...,...,...,...,...,...,...
1195,CTO-MG-BH-0176,OLT-MG-BH-B,MG-BH,Pampulha,Belo Horizonte,MG,-19.947488,-43.962977
1196,CTO-MG-BH-0177,OLT-MG-BH-A,MG-BH,Centro,Belo Horizonte,MG,-19.912985,-43.936935
1197,CTO-MG-BH-0178,OLT-MG-BH-B,MG-BH,Barreiro,Belo Horizonte,MG,-19.911402,-43.915058
1198,CTO-MG-BH-0179,OLT-MG-BH-A,MG-BH,Buritis,Belo Horizonte,MG,-19.921578,-43.942923


In [63]:
import json
meta = json.load(open("dados_telecom/_metadados.json", encoding="utf-8"))
esperado = meta["sujeira_injetada"]

achado = {
    "os_linhas_duplicadas": os_raw.count() - os_dedup.count(),
    "os_encerramento_antes_da_chegada": os_fix.filter("flag_datas_trocadas").count(),
    "os_concluida_sem_tecnico": os_silver.filter(
        "status = 'CONCLUIDA' and id_tecnico is null").count(),
    "assinante_linhas_duplicadas": ass_raw.count() - ass_dedup.count(),
}
for k, v in achado.items():
    ok = "OK" if v == esperado[k] else "DIVERGENTE"
    print(f"{k:38} esperado={esperado[k]:>5}  achado={v:>5}  [{ok}]")

os_linhas_duplicadas                   esperado=  600  achado=  600  [OK]
os_encerramento_antes_da_chegada       esperado=  133  achado=  133  [OK]
os_concluida_sem_tecnico               esperado=  399  achado=  399  [OK]
assinante_linhas_duplicadas            esperado=  173  achado=  173  [OK]


In [64]:
(os_silver.filter("conta_para_sla")
   .groupBy("id_regiao", "ano_mes")
   .agg(F.round(F.avg(F.col("dentro_sla").cast("int")) * 100, 1).alias("pct_no_sla"))
   .filter("id_regiao = 'SP-CAP-L'")
   .orderBy("ano_mes").show(20))

+---------+-------+----------+
|id_regiao|ano_mes|pct_no_sla|
+---------+-------+----------+
| SP-CAP-L|2025-09|      80.1|
| SP-CAP-L|2025-10|      79.3|
| SP-CAP-L|2025-11|      79.5|
| SP-CAP-L|2025-12|      73.6|
| SP-CAP-L|2026-01|      68.8|
| SP-CAP-L|2026-02|      42.1|
| SP-CAP-L|2026-03|      24.1|
| SP-CAP-L|2026-04|      55.3|
| SP-CAP-L|2026-05|      69.8|
| SP-CAP-L|2026-06|      71.2|
| SP-CAP-L|2026-07|      70.4|
| SP-CAP-L|2026-08|      71.6|
+---------+-------+----------+



In [65]:
gravar(ct, "contratos_sla")     # ct = contratos com os casts e datas do Passo 9

contratos_sla               30 linhas


c:\Users\filip\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,id_contrato,versao,id_cliente,codigo_tipo_servico,sla_horas,valor_os,multa_por_hora_atraso,teto_multa_pct,vigencia_inicio,vigencia_fim
0,CT-CLI01-INST_FTTH,1,CLI01,INST_FTTH,61,196.00,8.02,0.3,2024-09-01,None
1,CT-CLI01-REP_FTTH,1,CLI01,REP_FTTH,20,87.47,4.31,0.3,2024-09-01,2026-02-28
2,CT-CLI01-REP_FTTH,2,CLI01,REP_FTTH,12,87.47,6.46,0.3,2026-03-01,None
3,CT-CLI01-REP_LENT,1,CLI01,REP_LENT,41,90.73,3.95,0.3,2024-09-01,2026-02-28
4,CT-CLI01-REP_LENT,2,CLI01,REP_LENT,25,90.73,5.93,0.3,2026-03-01,None
5,CT-CLI01-MUD_END,1,CLI01,MUD_END,82,139.23,4.67,0.3,2024-09-01,None
6,CT-CLI01-RET_EQP,1,CLI01,RET_EQP,102,44.17,2.11,0.3,2024-09-01,None
7,CT-CLI01-MAN_PREV,1,CLI01,MAN_PREV,143,135.41,6.05,0.3,2024-09-01,None
8,CT-CLI01-REP_ROMP,1,CLI01,REP_ROMP,7,648.41,17.39,0.3,2024-09-01,None
9,CT-CLI02-INST_FTTH,1,CLI02,INST_FTTH,72,183.96,4.03,0.3,2024-09-01,None


In [66]:
cep_digitos = F.regexp_replace(F.col("cep"), r"\D", "")

ass_silver = (ass_dedup
    .withColumn("cep",
        F.when(F.length(cep_digitos) == 8,
               F.concat(F.substring(cep_digitos, 1, 5), F.lit("-"), F.substring(cep_digitos, 6, 3)))
         .otherwise(F.col("cep")))
    .withColumn("flag_cep_invalido", F.length(cep_digitos) != 8)
    .withColumn("nome_norm", F.initcap(F.trim(F.col("nome"))))
    .withColumn("plano_mbps", F.col("plano_mbps").cast("int"))
    .withColumn("data_ativacao", F.to_date("data_ativacao"))
    .withColumn("data_cancelamento", F.to_date("data_cancelamento"))
    .withColumn("flag_status_inconsistente",
        ((F.col("status") == "CANCELADO") & F.col("data_cancelamento").isNull()) |
        ((F.col("status") != "CANCELADO") & F.col("data_cancelamento").isNotNull()))
)
gravar(ass_silver, "assinantes")

print("sem hifen:", ass_silver.filter("cep not like '%-%'").count(), "(esperado 0)")
print("CEP invalido:", ass_silver.filter("flag_cep_invalido").count())
print("status incoerente:", ass_silver.filter("flag_status_inconsistente").count())

c:\Users\filip\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


assinantes               28779 linhas
sem hifen: 0 (esperado 0)
CEP invalido: 0
status incoerente: 0


In [67]:
# ============================================================
# PASSO 13 - Silver de alarmes_rede
# ============================================================
al = bronze["alarmes_rede"]

al_silver = (al
    .withColumn("dt_inicio", ts("dt_inicio"))
    .withColumn("dt_fim",    ts("dt_fim"))
    .withColumn("potencia_rx_dbm",   F.col("potencia_rx_dbm").cast("double"))
    .withColumn("qtd_onus_afetadas", F.col("qtd_onus_afetadas").cast("int"))
    .withColumn("flag_alarme_aberto",  F.col("dt_fim").isNull())
    .withColumn("flag_nivel_rede",     F.col("id_assinante").isNull())
    .withColumn("duracao_h", horas("dt_fim", "dt_inicio"))
    .withColumn("severidade_nivel",
        F.when(F.col("severidade") == "CRITICAL", 4)
         .when(F.col("severidade") == "MAJOR", 3)
         .when(F.col("severidade") == "MINOR", 2)
         .otherwise(1))
    .join(ctos.select("id_cto", "id_regiao"), on="id_cto", how="left")
    .withColumn("ano_mes", F.date_format("dt_inicio", "yyyy-MM"))
)

gravar(al_silver, "alarmes_rede", particoes=["ano_mes"])

print("abertos:", al_silver.filter("flag_alarme_aberto").count(), "(esperado 25)")
print("nivel rede:", al_silver.filter("flag_nivel_rede").count(), "(esperado 7054)")
print("duracao negativa:", al_silver.filter("duracao_h < 0").count(), "(esperado 0)")

c:\Users\filip\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\filip\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


alarmes_rede             31574 linhas
abertos: 25 (esperado 25)
nivel rede: 7054 (esperado 7054)
duracao negativa: 0 (esperado 0)


In [68]:
# ============================================================
# PASSO 14 - Silver de medicoes_potencia (438k linhas)
# ============================================================
med = (bronze["medicoes_potencia"]
    .withColumn("data", F.to_date("data"))
    .withColumn("potencia_rx_media_dbm", F.col("potencia_rx_media_dbm").cast("double"))
    .withColumn("potencia_rx_min_dbm",   F.col("potencia_rx_min_dbm").cast("double"))
    .withColumn("onus_total",  F.col("onus_total").cast("int"))
    .withColumn("onus_online", F.col("onus_online").cast("int"))
    .withColumn("pct_onus_online",
                F.round(F.col("onus_online") / F.nullif(F.col("onus_total"), F.lit(0)) * 100, 2))
)

# janelas por CTO ao longo do tempo
w7  = Window.partitionBy("id_cto").orderBy("data").rowsBetween(-6, 0)
w30 = Window.partitionBy("id_cto").orderBy("data").rowsBetween(-29, -7)

med_silver = (med
    .withColumn("pot_media_7d", F.round(F.avg("potencia_rx_media_dbm").over(w7), 2))
    .withColumn("pot_base_30d", F.round(F.avg("potencia_rx_media_dbm").over(w30), 2))
    # negativo = sinal piorando (dBm fica mais negativo)
    .withColumn("delta_dbm",    F.round(F.col("pot_media_7d") - F.col("pot_base_30d"), 2))
    .withColumn("flag_dia_degradando", F.col("delta_dbm") <= -0.8)
    .withColumn("ano_mes",      F.date_format("data", "yyyy-MM"))
)

# ---- esta linha fica FORA da cadeia, sem ponto na frente ----
gravar(med_silver, "medicoes_potencia", particoes=["ano_mes"])

print("linhas:", med_silver.count(), "(esperado 438000)")
print("dias degradando:", med_silver.filter("flag_dia_degradando").count())

c:\Users\filip\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


medicoes_potencia       438000 linhas
linhas: 438000 (esperado 438000)
dias degradando: 1922


In [69]:
import pandas as pd
for t in ["ordens_servico","assinantes","medicoes_potencia","alarmes_rede"]:
    d = pd.read_parquet(f"{SILVER}/{t}")
    print(f"{t:20} {len(d):>7} x {d.shape[1]}")

a = pd.read_parquet(f"{SILVER}/assinantes")
print("assinantes tipado:", dict(a[["plano_mbps","data_ativacao"]].dtypes.astype(str)))
m = pd.read_parquet(f"{SILVER}/medicoes_potencia")
print("dias degradando:", int(m.flag_dia_degradando.sum()), "(esperado ~2000-2500)")

ordens_servico         50000 x 45
assinantes             28779 x 17
medicoes_potencia     438000 x 12
alarmes_rede           31574 x 17
assinantes tipado: {'plano_mbps': 'int32', 'data_ativacao': 'object'}
dias degradando: 1922 (esperado ~2000-2500)
